In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Le decimos a Python que busque en la carpeta padre (donde está 'src')
sys.path.append(os.path.abspath('..'))

# Importamos nuestras funciones del proyecto modular
from src.parcial_ml import (
    riesgo_empirico,
    cargar_datos,
    entrenar_modelo_polinomico,
    experimento_para_un_dataset,
)

# Definimos las rutas
carpeta_datos = '../data'
carpeta_resultados = '../resultados'
os.makedirs(carpeta_resultados, exist_ok=True)

### EJERCICIO 1

a) Entrenar 15 modelos (grados 1 al 15) usando train_A, calcular el riesgo de cada uno sobre train_A y sobre validation, y mostrar la tabla.



In [ ]:
# Ejercicio 1(a): Riesgos para train_A con grados 1 al 15
grados = list(range(1, 16))

# Ejecutamos el experimento
resultados = experimento_para_un_dataset(
    carpeta_datos='../data',
    archivo_entrenamiento='train_A.csv',
    archivo_validacion='validation.csv',
    grados=grados,
    carpeta_resultados='../resultados'
)

# Mostramos la tabla con las columnas que pide el examen
resultados[['Degree', 'Train_Risk', 'Val_Risk']]


Al aumentar el grado, el riesgo de entrenamiento baja constantemente, pero el de validación alcanza su mínimo en el grado 5 (0.1614) y luego comienza a subir. Este comportamiento indica sobreajuste en grados altos. El grado 5 es el mejor balance entre ajuste y generalización.


b) Grafique, en una misma figura, el riesgo de entrenamiento y el riesgo de validación en función
del grado.


In [ ]:
plt.figure(figsize=(10, 6))

# Curva del riesgo de entrenamiento (azul)
plt.plot(resultados['Degree'], resultados['Train_Risk'],
         marker='o', color='royalblue', linewidth=2,
         label='Riesgo de Entrenamiento')

# Curva del riesgo de validación (rojo)
plt.plot(resultados['Degree'], resultados['Val_Risk'],
         marker='s', color='crimson', linewidth=2,
         label='Riesgo de Validación')

plt.xlabel('Grado del polinomio $d$', fontsize=12)
plt.ylabel('Riesgo empírico $\\hat{R}_S(h)$', fontsize=12)
plt.title('Ejercicio 1(b): Complejidad vs Generalización (train_A)', fontsize=13)
plt.xticks(grados)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Guardamos la gráfica
plt.savefig(os.path.join(carpeta_resultados, 'ej1b_grafica.png'), dpi=150)
plt.show()

La gráfica muestra que el riesgo de entrenamiento (azul) disminuye constantemente al aumentar el grado, mientras que el riesgo de validación (rojo) alcanza su mínimo en el grado 5 y luego comienza a subir. La separación creciente entre ambas curvas a partir del grado 7 evidencia sobreajuste.

c) busca en la tabla cuál grado tiene el riesgo más pequeño en cada columna.

In [ ]:
d_train = int(resultados.loc[resultados['Train_Risk'].idxmin(), 'Degree'])
d_val   = int(resultados.loc[resultados['Val_Risk'].idxmin(), 'Degree'])
riesgo_train_min = resultados['Train_Risk'].min()
riesgo_val_min   = resultados['Val_Risk'].min()

print(f"d_train = {d_train}   =>  riesgo mínimo de entrenamiento: {riesgo_train_min:.6f}")
print(f"d_val   = {d_val}   =>  riesgo mínimo de validación:    {riesgo_val_min:.6f}")

El modelo que mejor se aprende los datos de memoria es el de grado 12, pero cuando le pedimos predecir datos nuevos, falla. En cambio, el grado 5 no memoriza tan bien, pero es mucho más confiable para datos nuevos. Como en la vida real queremos que el modelo funcione con datos que nunca ha visto, el grado 5 es la mejor opción.

d)Solo grados 2, 6 y 15

In [ ]:
tabla_d = resultados[resultados['Degree'].isin([2, 6, 15])][['Degree', 'Train_Risk', 'Val_Risk']]
tabla_d

El grado 2 presenta subajuste: ambos riesgos son altos (train 0.33, validación 0.44) porque el modelo es demasiado simple para capturar el patrón. El grado 6 logra el mejor balance: riesgos bajos y similares (train 0.08, validación 0.16), lo que indica buena generalización. El grado 15 sufre sobreajuste: aunque su riesgo de entrenamiento es bajo (0.13), el de validación es alto (0.33), señal de que memoriza el ruido y falla con datos nuevos.

### EJERCICIO 2

a) Para cada grado construya una gráfica que muestre simultáneamente las tres hipótesis aprendidas. Deben obtenerse tres figuras en total.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.parcial_ml import analizar_sensibilidad

grados_ej2 = [2, 6, 15]

resultados_ej2 = analizar_sensibilidad(
    carpeta_datos='../data',
    grados=grados_ej2,
    carpeta_resultados='../resultados',
    m=300
)

for grado in [2, 6, 15]:
    print(f"V_{grado} = {resultados_ej2[grado]['V_d']:.6f}")

En el grado 2, las 3 hipótesis son casi idénticas: el modelo no depende de la muestra porque es demasiado simple. En el grado 6, empiezan a notarse diferencias: es el punto intermedio. En el grado 15, las curvas son completamente distintas: cada modelo memoriza el ruido específico de su muestra, lo cual se reflejará en una varianza alta. Esto confirma que a mayor complejidad, mayor sensibilidad a la muestra, y que un modelo estable es preferible aunque no sea el más preciso en entrenamiento.